# Working with the Anthropic API  

This notebook introduces the **Anthropic Python SDK** and the **Messages API**. We will cover the essentials of authenticating with Claude, sending prompts, and building simple multi-turn conversations.

Along the way, we will explore:
- initializing the Anthropic client
- sending messages to Claude
- maintaining conversation history
- controlling response length and structure
- using assistant-message prefilling
- handling model outputs safely

The goal is to understand the core patterns used to build applications with Claude, starting with a basic API request and progressing to a small conversational app.


##  Setup  

Before making our first request to Claude, we need to prepare the Python environment and initialize the Anthropic client.

This section covers three steps:

- **Import the Anthropic SDK**: provides the Python interface for interacting with the Anthropic API.  
- **Load the API credentials**: retrieves the API key from the environment rather than storing it directly in the notebook.  
- **Initialize the client**: creates the Anthropic client that we will use throughout the notebook.  


### Import SDK  
 The anthropic package provides the Python SDK for interacting with Claude through the Anthropic API. 

In [ ]:
from anthropic import Anthropic

### Load credentials  
  
This repository uses environment variables to keep the `Anthropic API key` separate from the code. The provided helper function (`helper.py`) loads the variables from the local `.env` file (see `.env_example` file).  

In [ ]:
# Load environment variables ANTHROPIC_API_KEY
from helper import load_env
load_env()

### Initialize client  

Once the environment variable has been loaded, an `Anthropic client` should be created. This Anthropic client will use the `Anthropic API key` environment variable when making API requests.


In [ ]:
# Initialize the Anthropic client
client = Anthropic()

## Select a model  

The Anthropic API provides access to [different Claude models](https://platform.claude.com/docs/en/models/overview), each designed for different types of tasks and workloads.  

The model identifier is stored in a variable so that it can be reused throughout the notebook.

In [ ]:
# Example model configuration.
MODEL_NAME="claude-sonnet-5"  # Update `MODEL_NAME` if necessary 

## Making an API call  

The `messages.create()` method is the core API call. It accepts several paramenters that are shown in the following table:  


| Parameter    | Purpose                                  |
| ------------ | ---------------------------------------- |
| `model`      | Selects the Claude model                 |
| `max_tokens` | Sets the maximum number of output tokens |
| `messages`   | Contains the conversation                |

  
_Note_: Each message includes a `role` and `content`.  

### Make a request  

Let's start with a simple prompt and ask Claude to write a haiku about Anthropic.

In [ ]:
# A basic prompt request: ask Claude to write a haiku
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "Write a haiku about Anthropic"}
    ]
)


### Inspecting the complete response  

- For a simple text response, the generated text can be accessed directly from the first content block:

In [ ]:
print(response.content[0].text)

- If you want to explore the structure returned by the API, you can inspect the complete response object:

In [ ]:
response

**Explanation output** : As it can been seen, the response is a structured object containing metadata, such as the message ID, and content blocks. In this notebook, we print only the blocks that actually contain text. 

## The Messages Format  

Anthropic's Messages API uses a list of messages, where each message is a dictionary with:  

- `role`: either "user" or "assistant"  
- `content`: the text or structured content for that turn 

| Role | Meaning |
|------|---------|
| `user` | A message from the user/application |
| `assistant` | A previous response from Claude |


This format is fundamental for multi-turn conversations.  

The Messages API is stateless at the request level. To continue a conversation, the application sends previous messages again as part of the `messages` list.

 This example shows how to keep conversation context across turns.

In [ ]:
# The assistant is told to respond in Dutch.
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "I am learning about Anthropic. Please answer in Dutch."},
        {"role": "assistant", "content": "Natuurlijk! Ik zal in het Nederlands antwoorden."},
        {"role": "user", "content": "What is Anthropic?"}
    ]
)


# The response object contains content blocks.
# A robust way to print the generated text:
for block in response.content:
    if hasattr(block, "text"):
        print(block.text)


**Explanation output** : The output shows how conversation context is preserved. Once the assistant has responded earlier, the next user message can be interpreted in context.

##  Building a simple chatbot  

A chatbot can be built by storing the conversation history in a list of messages. Each time the user enters a new message, it is appended to the list, and the model generates a reply based on the whole conversation so far.  


### From individual requests to a conversation

So far, each API call has been created manually. We can now combine what we have learned to build a simple conversational loop.

The key idea is:

1. Get input from the user.
2. Add it to the message history.
3. Send the history to Claude.
4. Add Claude's response to the history.
5. Repeat.

In [ ]:
print("Simple Chatbot (type 'quit' to exit)")
# Store conversation history
messages = []
while True:
    # Get user input
    user_input = input("You: ")
    # Check for quit command
    if user_input.lower() == 'quit':
        print("Goodbye!")
        break
    # Add the user's latest message to the conversation history
    messages.append({"role": "user", "content": user_input})
    try:
        # Get response from Claude
        response = client.messages.create(
            model=MODEL_NAME,
            max_tokens=200,
            messages=messages
        )
        # Extract and print Claude's response
        assistant_message = response.content[0].text
        print("Assistant:", assistant_message)
        
        # Store the assistant response so the model can continue the exchange
        messages.append({"role": "assistant", "content": assistant_message})
        
    except Exception as e:
        print(f"An error occurred: {e}")

This pattern is common in real-world chatbot applications, because it allows the model to remember prior turns and respond in a consistent style.

## Prefilling the Assistant Response  

When continuing a draft, shaping the tone, or maintaining a specific expected direction, there is the option to guide the assistant by providing a pre-written assistant message.

In [ ]:
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "Write a short poem about Dutch people"},
        {"role": "assistant", "content": "Oink"},
        {"role": "user", "content": "Continue the poem in a more polished, poetic style."}
    ]
)

#print(response.content[0].text)

for block in response.content:
    if hasattr(block, "text"):
        print(block.text)

**Explanation output** : This example shows that an assistant message does not always have to be generated by the model in the same call. It can be included explicitly to steer the next output.  

## Model Parameters    

The API supports several generation parameters to guide the model's output length and shape. The most relevant parameters are: 

- `max_tokens`  
- `stop_sequences`  
- [other Claude models](https://platform.claude.com/docs/en/models/overview), each designed for different types of tasks and workloads.  

### Max Tokens  

`max_tokens` limits the total length of the model's output and it is the simplest ways to keep responses concise or allow longer output when needed.

In [ ]:
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=100,
    messages=[
        {"role": "user", "content": "Write me an essay on LLMs"},
    ]
)
print(response.content[0].text)

In [ ]:
response

**Explanation output** : The output was truncated because the model stopped for reaching the output token limit — the Message shows stop_reason='max_tokens' and usage.output_tokens=100, so the generation hit a 100-token cap and stopped mid-word.

The issue of the prompt that is been cut off, can be solved for instance, try to short the prompt / system messages so more of the context window is available for output,using the same number of tokens.

In [ ]:
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=100,
    messages=[
        #{"role": "user", "content": "Write me a short overview on LLMs"},
        {"role": "user", "content": "In one sentence write me what LLMs"},
    ]
)
print(response.content[0].text)

In [ ]:
response

### Stop Sequences  

`stop_sequences` tells the model to stop generating once it encounters a specific string. This is especially useful when the objective is to have a structured output, such as a numbered list that should terminate at a certain point.

Let's consider the following example in which the model is asked to produce a numbered list:

In [ ]:
prompt = """
Generate a numbered, ordered list of technical topics 
I should learn if I want to work on LLMs
"""
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=500,
    messages=[{"role": "user", "content": prompt}],
)
print(response.content[0].text)

To avoid the truncation of the output and further to have an output concise or bounded, it is possible to stop the generation  as soon as the sequence, for instance "4." appears (`stop_sequences`= [4]). 

In [ ]:
prompt = """
Generate a numbered, ordered list of technical topics 
I should learn if I want to work on LLMs
"""
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=500,
    stop_sequences=["4."],
    messages=[{"role": "user", "content": prompt}],
)
print(response.content[0].text)

In [ ]:
response

## Controlling response variability with prompting  

Earlier Claude models, such as `claude-3-5-sonnet-20241022`exposed sampling parameters such as `temperature`, which could be used to influence the variability of generated responses.

`Claude Sonnet 5` does not support non-default values for `temperature`, `top_p`, or `top_k`. Instead, **Anthropic recommends using prompting to guide the model's behavior**.

We can therefore experiment with the level of variation explicitly through instructions in the prompt.

### Prompting for consistency vs. creativity

In [ ]:
def ask_claude(prompt):
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=500,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.content[0].text

In [ ]:
consistent_prompt = """
Write a two-sentence description of a deep learning model.

Be concise, factual, and consistent.
Use exactly two sentences.
Do not use figurative language.
"""

In [ ]:
creative_prompt = """
Write a two-sentence description of a deep learning model.

Be creative and imaginative.
Use vivid language and different imagery.
Avoid generic phrases.
"""

In [ ]:
print("=== CONSISTENT ===")
for _ in range(3):
    print(ask_claude(consistent_prompt))
    print()

print("=== CREATIVE ===")
for _ in range(3):
    print(ask_claude(creative_prompt))
    print()

**Explanation output** :The output shows two prompting styles and their effects — the "CONSISTENT" outputs are near‑identical, factual two‑sentence descriptions, while the "CREATIVE" outputs are varied, metaphorical, and more expressive.

## Conclusion  

This notebook demonstrates the Anthropic Messages API—initializing the client, composing user/assistant messages, keeping conversation history, building a simple chatbot, and controlling output—so that the message‑based patterns can be reused in the multimodal and tool‑use notebooks.

## Practice challenges

The following challenges are designed to reinforce the main concepts from this notebook. 


###  Challenge 1 : Conversation memory

To understand how message history provides context, build a chatbot that remembers information provided earlier in the conversation.

In [ ]:
## Snippet code example
messages = [
    {"role": "user", "content": "My favorite topic is Anthropic"},
    {"role": "assistant", "content": "Nice! I will remember that."}
]

response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=200,
    messages=messages + [
        {"role": "user", "content": "Explain the most important discoveries in Anthropic."}
    ]
)

### Challenge 2 : Prompt comparison

To observe how instructions affect the response, Ask the same question using four different prompting styles such as:  
- direct question  
- detailed prompt  
- role-based prompt  
- step-by-step instruction  


In [ ]:
## Snippet code example
prompts = [
    "Summarize machine learning in one paragraph.",
    "Explain machine learning to a beginner in simple language.",
    "Act as a teacher and explain machine learning with examples.",
    "Give me a short list of 5 key ideas in machine learning."
]

for prompt in prompts:
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}]
    )
    print("PROMPT:", prompt)
    for block in response.content:
        if hasattr(block, "text"):
            print(block.text)
    print("\n" + "=" * 50 + "\n")

###  Challenge 3 : Structured output

To move from text generation toward application-ready output, ask Claude to produce structured JSON and investigate how you could validate and parse the result in Python.

In [ ]:
## Snippet code example
prompt = """
Return a JSON object with:
- name
- purpose
- benefits
- example_use_case

The answer must be valid JSON only.
"""

response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=300,
    messages=[{"role": "user", "content": prompt}]
)

for block in response.content:
    if hasattr(block, "text"):
        print(block.text)

### Challenge 4 : Output limits

To understand how output limits affect generation, deploy the same prompt with different `max_tokens` values and compare the results.

In [ ]:
## Snippet code example
for tokens in [50, 150, 300]:
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=tokens,
        messages=[{"role": "user", "content": "Write a short explanation of Anthropic"}]
    )
    print(f"MAX_TOKENS = {tokens}")
    for block in response.content:
        if hasattr(block, "text"):
            print(block.text)
    print("=" * 40)

###  Challenge 5 : Error handling

To learn how to build more resilient applications, inspect the outputs that come from experimenting with invalid or unusual requests, such as:

- empty prompts  
- very long prompts  
- very short outputs  
- ask for unsupported formatting  
- ask for a list with unusual constraints  


In [ ]:
## Snippet code example  
try:
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=50,
        messages=[{"role": "user", "content": ""}]
    )
except Exception as e:
    print("Error:", e)

###  Challenge 6 : Build your application 

Combine the concepts from this notebook to create a small application:

- study assistant
- writing assistant
- language tutor
- topic explainer
- Q&A bot



In [ ]:
## Snippet code example
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=300,
    messages=[
        {"role": "user", "content": "You are a careful research assistant. Explain the difference between AI, ML, and deep learning in 3 bullet points."}
    ]
)
for block in response.content:
    if hasattr(block, "text"):
        print(block.text)